# 03 · Error analysis

**Task T8** in [TASKS.md](../TASKS.md). One F1 number says how good a model is. This notebook looks at *what* it gets wrong and *why*.

It needs no GPU and runs no training. It reads the predictions that `src/evaluate.py` saved for every dev question (`results/*_predictions.json`), so the first four sections take a few seconds. Only the last section loads the models.

1. What kind of mistake is each wrong prediction? (automatic, all 11,873 questions)
2. Which question words are hard?
3. Do long answers hurt?
4. 60 wrong predictions labelled by hand: why did it go wrong?
5. An example from outside the dataset

All the code is in `src/errors.py`.

## 0. Setup
Clones the repo and installs the requirements in Colab. Safe to run more than once.

In [ ]:
import os, sys

REPO_URL = "https://github.com/Davidfaleye360/AI-Question-Answering-System.git"
REPO_DIR = "/content/AI-Question-Answering-System"

if "google.colab" in sys.modules:
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q
    %cd {REPO_DIR}
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # running locally: work from the repo root

sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

## 1. What kind of mistake?

Every dev question ends in exactly one of these outcomes, decided automatically by comparing the prediction with the gold answers:

| Outcome | Meaning | Example |
| --- | --- | --- |
| correct | exact match | |
| false answer | the question is unanswerable, but the model gave a span | |
| missed answer | the question is answerable, but the model said "no answer" | |
| too long | the right answer plus extra words | "Richard I of Normandy" for "Richard I" |
| too short | only part of the right answer | "Nobel Prize" for "Nobel Prize in Physics" |
| partial overlap | shares words with the answer, but neither contains the other | |
| wrong span | shares no words with any gold answer | "Christian" for "Catholic" |

The next cell does the counting for both models and saves `results/error_breakdown.json` and `results/error_analysis.png`.

In [ ]:
!{sys.executable} -m src.errors --models distilbert-squad2 roberta-base-squad2 --labels "DistilBERT (ours)" "RoBERTa (deepset)"

In [ ]:
import json
import pandas as pd

from src.errors import ERROR_TYPES

breakdown = json.load(open("results/error_breakdown.json"))["models"]
NAMES = {"distilbert-squad2": "DistilBERT (ours)", "roberta-base-squad2": "RoBERTa (deepset)"}

table = pd.DataFrame({
    f"{label}: questions": [breakdown[name]["outcomes"][e] for e in ERROR_TYPES]
    for name, label in NAMES.items()
}, index=list(ERROR_TYPES))
for name, label in NAMES.items():
    table[f"{label}: % of its errors"] = [breakdown[name]["error_shares_pct"][e] for e in ERROR_TYPES]
table.loc["all errors"] = [breakdown[n]["errors"] for n in NAMES] + [100.0, 100.0]
table

**What this shows**

- Three out of four of our model's errors are about the **answer / no-answer decision**, not about finding the span: 52% are false answers and 24% are missed answers.
- When it does answer an answerable question, it is usually in the right place. "Too long" and "too short" together are 13% of its errors, and section 4 shows most of those would be accepted by a person.
- RoBERTa makes about half as many errors, and the biggest saving is in false answers (1,081 instead of 2,224). Being better at saying "no answer" is most of the gap between the two models.

## 2. Which question words are hard?
F1 on **answerable** questions only, so "did it find the right span" is not mixed up with "did it abstain when it should".

In [ ]:
def side_by_side(key, column):
    frames = []
    for name, label in NAMES.items():
        frame = pd.DataFrame(breakdown[name][key]).set_index(column)
        frames.append(frame.rename(columns={"f1": f"F1, {label}", "exact": f"EM, {label}"}))
    merged = frames[0].join(frames[1].drop(columns="questions"))
    merged["F1 gap"] = (merged[f"F1, {NAMES['roberta-base-squad2']}"] - merged[f"F1, {NAMES['distilbert-squad2']}"]).round(1)
    return merged

side_by_side("by_question_type_answerable", "question_type")

**What this shows**

- Questions that ask for a **number or a date** are the easiest: "how many" (85 F1) and "when" (84). The answer is short and has an obvious shape.
- **"Why" and "how"** are the hardest (59 and 64): their answers are explanations, which are long and have fuzzy edges. RoBERTa struggles with "why" too (70), so this is partly the task and not only our model.
- "What" questions are 63% of the dataset and score 70, so they decide the overall number.

## 3. Do long answers hurt?
Answerable questions grouped by the length of the (shortest) gold answer.

In [ ]:
side_by_side("by_answer_length", "answer_length")

**What this shows**

Up to four words, length makes little difference (71 to 74 F1). From five words the score falls, and for answers of ten or more words our model drops to 51 F1 while RoBERTa holds 73. Long answers are rare in training (the median answer is two words), and every extra word is another chance to cut the span in the wrong place.

The figure below puts the three views side by side.

In [ ]:
from IPython.display import Image

Image("results/error_analysis.png")

## 4. Why did it go wrong? 60 errors labelled by hand

The automatic types say *what* happened. To learn *why*, `src/errors.py` draws a fixed random sample of 60 of our model's wrong predictions (every error type is represented) into `results/error_sample.csv`, and each one gets a label after reading the question, the gold answers, the prediction and the passage.

| Label | Meaning |
| --- | --- |
| swapped detail | An unanswerable question that changes one detail (a name, a date, the subject). The model answers the original fact. |
| reversed meaning | An unanswerable question that flips the meaning (united → fought, older → younger). The model ignores the flip. |
| not stated | The passage simply doesn't say; the model picks something of the right type. |
| gold label debatable | The model's answer is defensible. |
| over-cautious, had the right span | The model said "no answer", but its best span was correct. |
| missed, best span also wrong | It said "no answer", and its best span was wrong as well. |
| boundary, answer acceptable | Right answer, slightly different edges ("pine forest" for "pine"). A person would accept it. |
| distractor | A wrong span that looks right: another name, date or phrase that fits the question's wording. |
| needs reasoning | The answer needs linking two statements, not just matching words. |

The labels were proposed during development and then read by the team. To change one, edit the `label` or `note` column of the CSV; re-running section 1 keeps what is in those two columns.

**Columns of `results/error_sample.csv`**

| Column | What it holds |
| --- | --- |
| `id` | The question's id in SQuAD, to look it up in the dataset. |
| `error_type` | The automatic type from section 1 (false answer, missed answer, too long, ...). |
| `question` | The question as asked. |
| `correct_answers` | The correct answers written by SQuAD's annotators, separated by `\|`, or `(no answer)` if the question is unanswerable. |
| `prediction` | What our model answered, or `(no answer)` if it abstained. |
| `best_span_if_abstained` | Filled only when the model abstained: the answer it would have given if forced to. |
| `score_diff` | The no-answer score minus the best span's score. Negative: the model preferred the span. Positive: it preferred "no answer". Close to 0: it was unsure. |
| `passage_near_answer` | The part of the passage around the correct answer (empty for unanswerable questions). |
| `passage_near_prediction` | The part of the passage around the model's answer or best span. |
| `label` | Why it went wrong, from the list above. Written by hand. |
| `note` | One sentence explaining the label for this example. Written by hand. |

In [ ]:
from src.errors import label_summary

summary = label_summary("results/error_sample.csv")
print(f"{summary['labelled']} errors labelled\n")
pd.DataFrame(summary["by_error_type"]).fillna(0).astype(int).assign(total=lambda t: t.sum(axis=1)).sort_values("total", ascending=False)

In [ ]:
sample = pd.read_csv("results/error_sample.csv").fillna("")
pd.set_option("display.max_colwidth", 110)
sample[["error_type", "question", "correct_answers", "prediction", "best_span_if_abstained", "label", "note"]]

**What this shows** (counts are out of the 60 sampled errors, not percentages of all errors)

- **False answers (27):** 24 of them are trick questions that reuse the passage's wording and change one thing. 14 swap a detail and 10 reverse the meaning. The model matches the wording and misses the change. This is the single biggest weakness.
- **Missed answers (15):** in 10 the model's best span was the right answer and it abstained anyway. Many of these questions are worded far from the passage or contain typos. These are the cases a higher threshold would win back, at the price of more false answers, which is exactly the trade-off measured in T7.
- **Too long / too short (9):** all 9 are answers a person would accept ("c. 1,300,000" for "1,300,000", "Karakorum in Mongolia" for "Karakorum"). Exact match counts them as wrong, so EM understates how often the model is right; F1 gives them partial credit.
- **Wrong span (5) and partial overlap (4):** 7 are distractors, typically a second date or name that fits the same phrase (two "Commission v Italy" cases, two people who "became Great Khan").
- Only 1 of the 60 needed real reasoning. In SQuAD most errors come from surface matching, not from questions that are deep.

## 5. An example from outside the dataset

David tried this passage in the demo. The question asks *why*, and the reason is never stated in one place: a reader has to connect Professor Aris's argument with the board's decision two sentences later. By SQuAD's rules the question is arguably unanswerable.

The next cell runs both models on it (this downloads them the first time, about 770 MB together).

In [ ]:
from src.config import load_config
from src.demo import MODELS, ask, get_model, render

PASSAGE = ("The restoration of the 14th-century fresco was heralded as a triumph of modern conservation, though it "
           "was not without intense debate. Dr. Vance argued that the vibrant cobalt hues revealed by the laser "
           "scaling were exactly what the original master intended, pointing to contemporary trade logs from "
           "Florence. However, his colleague, Professor Aris, maintained that the sudden brilliance was merely an "
           "artifact of the chemical solvent interacting with later resin layers applied during the 17th century. "
           "While the museum board ultimately voted to halt the chemical treatments, they allowed the physical "
           "scraping to proceed under strict supervision, leaving the final appearance of the saint's robe "
           "permanently caught between two eras.")
QUESTION = "Why did the museum board decide to stop using the chemical treatments?"

cfg = load_config("configs/distilbert.yaml")
for label, info in MODELS.items():
    model, tokenizer = get_model(label)
    result = ask(model, tokenizer, QUESTION, PASSAGE, cfg)
    answer = render(result, info.default_tau)[0].lstrip("# ")
    print(f"{label}\n  answer: {answer}\n  best span beats 'no answer' by {-result['score_diff']:.2f}\n")

Both models give the clause that sits next to the words of the question ("they allowed the physical scraping to proceed...") and neither abstains. The much stronger RoBERTa fails in the same way, so this is a limit of the method, not of our training:

- An **extractive** model can only copy one continuous piece of the passage. It cannot combine two sentences or write an explanation.
- It has learned that the answer usually sits near the question's words. That is true in SQuAD and false here.

## Takeaways for the report

1. Three quarters of our errors are answer / no-answer decisions; span finding is the smaller problem.
2. Trick questions that change one detail or reverse the meaning cause most false answers.
3. EM understates quality: the boundary errors we read were all acceptable answers.
4. "Why" and "how" questions and answers longer than about five words are the hard cases, for both models.
5. Questions that need reasoning across sentences are outside what an extractive model can do.